# CoroVision AI · ARCADE instance-aware training
This notebook trains Mask R-CNN models from the original COCO polygons and category labels. Syntax (25 vessel classes) and stenosis (stenosis category) are separate tasks. Checkpoint selection uses the public ARCADE validation F1 rules; instance-matched and binary-union pixel metrics are reported as supplements. The held-out test set is never included in this training bundle. Research outputs only; not clinical validation.

Colab uses its GPU for Mask R-CNN forward/backward passes. Colab CPU workers decode and augment images, while CPU code handles polygon scoring and metric aggregation. Choose **Runtime → Change runtime type → T4 GPU**. Upload `corovision_arcade_instance_bundle.zip` to `MyDrive/CoroVisionAI/`, then run the cells in order. Training can take many GPU hours.

In [ ]:
import os, torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| CPU cores:', os.cpu_count())
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Runtime → Change runtime type, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), '| CPU data-loader workers will be enabled')


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/CoroVisionAI')
BUNDLE = DRIVE_DIR / 'corovision_arcade_instance_bundle.zip'
assert BUNDLE.is_file(), f'Upload the train/validation-only bundle to {DRIVE_DIR}'
print(f'Bundle size: {BUNDLE.stat().st_size / 1024**2:.1f} MiB')


In [ ]:
import shutil, zipfile
PROJECT = Path('/content/corovision')
if PROJECT.exists(): shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert not any('/test/' in n for n in z.namelist()), 'Test data must not be in the training bundle'
    z.extractall(PROJECT)
TRAIN_SCRIPT = PROJECT / 'scripts/train_arcade_instances.py'
for task in ('syntax', 'stenosis'):
    for split in ('train', 'val'):
        root = PROJECT / 'arcade' / task / split
        assert (root / 'annotations' / f'{split}.json').is_file()
        assert (root / 'images').is_dir()
        print(task, split, 'ready')
print('Test split excluded.')


In [ ]:
%pip -q install 'shapely>=2,<3' 'opencv-python-headless>=4.9,<5'
import torchvision
print('TorchVision:', torchvision.__version__)
print('Instance-aware training and official-metric evaluator will run as a subprocess.')


In [ ]:
# Full category-aware experiment. CPU workers handle data preparation; CUDA handles model training.
from datetime import datetime, timezone
import subprocess, sys
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
CPU_WORKERS = max(2, min(8, (os.cpu_count() or 3) - 1))
EXPERIMENT_DIR = f'models/arcade_instances/experiments/{RUN_ID}'
cmd = [sys.executable, str(TRAIN_SCRIPT), '--task', 'both',
       '--architectures', 'maskrcnn_resnet50_fpn', 'maskrcnn_resnet50_fpn_v2',
       '--epochs', '25', '--batch-size', '2', '--input-size', '512',
       '--patience', '6', '--workers', str(CPU_WORKERS), '--seed', '42',
       '--run-id', RUN_ID, '--output-dir', EXPERIMENT_DIR]
print('Run:', RUN_ID, '| GPU:', torch.cuda.get_device_name(0), '| CPU workers:', CPU_WORKERS)
p = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout: print(line, end='')
if p.wait() != 0: raise RuntimeError('Training failed; inspect the log above.')


In [ ]:
# Review the validation-ranked candidate. It is not deployed or auto-promoted.
import json
MODEL_DIR = PROJECT / EXPERIMENT_DIR
selected = json.loads((MODEL_DIR / 'candidate.json').read_text())
print(json.dumps(selected, indent=2))
assert selected['test_used_for_training_or_selection'] is False


In [ ]:
# Export candidate checkpoints, reports, and checksums to Drive for review.
import hashlib
selected_arch = selected['architecture']
files = [MODEL_DIR / 'candidate.json', MODEL_DIR / 'comparison_report.json']
files += [MODEL_DIR / task / selected_arch / 'best.pt' for task in ('syntax', 'stenosis')]
files += [MODEL_DIR / task / selected_arch / 'report.json' for task in ('syntax', 'stenosis')]
manifest = {'run_id': RUN_ID, 'candidate_architecture': selected_arch, 'validation_official_f1_by_architecture': selected['validation_official_f1_by_architecture'], 'test_used_for_training_or_selection': False, 'clinical_validation': False, 'sha256': {str(p.relative_to(PROJECT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}}
out = DRIVE_DIR / f'corovision_instance_candidate_{RUN_ID}.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files: z.write(p, p.relative_to(PROJECT))
    z.writestr('models/arcade_instances/export_manifest.json', json.dumps(manifest, indent=2))
print('Saved', out, f'({out.stat().st_size / 1024**2:.1f} MiB)')
print(json.dumps(manifest, indent=2))


## Review and one-time held-out test
The exported files are a validation-ranked research candidate. They do not replace the active model automatically. Review the saved official-F1 results, per-class scores, overlays, and failure cases first.

After choosing the candidate and restoring the full original ARCADE dataset (including test images and labels) under `/content/corovision/arcade/`, run the test-only evaluation once, passing the exact experiment directory: `python scripts/train_arcade_instances.py --evaluate-test-only --output-dir models/arcade_instances/experiments/<RUN_ID>`. This reads test only for final reporting; it does not train or tune. Do not use test results to select another candidate. The public evaluator should be cross-checked before any challenge submission.

For later application integration, promote the reviewed candidate as a separate deliberate step; the current app model and previous checkpoints remain untouched by this notebook.